## 1. `depth_chart`
DECISION: Remove table, does not have weekly granularity needed - get from sleeper or alt source

### ✂️ Suggestions — `depth_chart`

**Keep all 7 columns.** This table is already minimal — every column carries distinct meaning.

| Column | Keep? | Reason |
|---|---|---|
| `gsis_id` | ✅ | Primary join key |
| `dc_team` | ✅ | Current team (snapshot) |
| `dc_pos_abb` | ✅ | Abbreviated position (join-friendly) |
| `dc_pos_slot` | ✅ | Depth slot (starter vs backup) |
| `dc_pos_rank` | ✅ | Rank within position group |
| `dc_pos_name` | ✅ | Verbose position name (human-readable) |
| `dc_pos_grp` | ✅ | Offensive formation grouping |

## 2. `fantasy_ids`
DECISION: Keep table and all columns for future data integrations

### ✂️ Suggestions — `fantasy_ids`

This table's **only modeling purpose** is bridging `pfr_player_id ↔ gsis_id`. All other platform IDs are unused downstream.

| Column | Keep? | Reason |
|---|---|---|
| `gsis_id` | ✅ | Primary key for most tables |
| `pfr_player_id` | ✅ | Primary key for snap_counts and pfr_adv_stats |
| `sportradar_id` | ❌ | Platform ID not used in any join |
| `espn_id` | ❌ | Platform ID not used in any join |
| `sleeper_id` | ❌ | Platform ID not used in any join |
| `yahoo_id` | ❌ | Platform ID not used in any join |
| `mfl_id` | ❌ | Platform ID not used in any join |

**Action:** Remove `sportradar_id`, `espn_id`, `sleeper_id`, `yahoo_id`, `mfl_id` from `loaders/fantasy_ids.py`.

## 3. `fantasy_opportunities`
DECISION: Remove this table. not reliably uploaded and also not weekly released.

## 4. `fantasy_rankings`
DECISION: remove table for now, does not have a weekly granularity just does a live scrape

## 5. `formations`
DECISION: keep, good weekly team tendnency data

### ✂️ Suggestions — `formations`

**Keep all 10 columns.** This table is clean — all columns are distinct modeling features with no redundancy.

| Column | Keep? | Reason |
|---|---|---|
| `game_id` | ✅ | Join key |
| `gsis_id` | ✅ | Join key |
| `team` | ✅ | Context |
| `form_off_snaps` | ✅ | Snap count (denominator for pcts) |
| `form_shotgun_pct` | ✅ | Key formation signal |
| `form_under_center_pct` | ✅ | Complement to shotgun |
| `form_empty_back_pct` | ✅ | Passing-tendency signal |
| `form_pressure_rate` | ✅ | QB protection signal |
| `form_avg_time_to_throw` | ✅ | QB pocket mobility |
| `form_avg_defenders_box` | ✅ | Run/pass tendency of defense |

## 6. `nextgen`
DECISION: keep - good weekly data, only keep the columns noted under suggestions

### ✂️ Suggestions — `nextgen`

The unique NextGen value is **tracking-derived quality metrics** (separation, CPAE, RYOE). Basic counting stats duplicate `weekly_player_stats`.

**Drop — duplicate counting stats (already in `weekly_player_stats`):**

| Column | Drop Reason |
|---|---|
| `ng_pass_attempts` | = `weekly_player_stats.passing_attempts` |
| `ng_pass_pass_yards` | = `weekly_player_stats.passing_yards` |
| `ng_pass_pass_touchdowns` | = `weekly_player_stats.passing_tds` |
| `ng_pass_interceptions` | = `weekly_player_stats.interceptions` |
| `ng_pass_passer_rating` | Derivable from other stats; high correlation with CPAE |
| `ng_rush_rush_attempts` | = `weekly_player_stats.rushing_attempts` |
| `ng_rush_rush_yards` | = `weekly_player_stats.rushing_yards` |
| `ng_rush_rush_touchdowns` | = `weekly_player_stats.rushing_tds` |
| `ng_rec_receptions` | = `weekly_player_stats.receptions` |
| `ng_rec_targets` | = `weekly_player_stats.targets` |
| `ng_rec_rec_yards` | = `weekly_player_stats.receiving_yards` |
| `ng_rec_rec_touchdowns` | = `weekly_player_stats.receiving_tds` |
| `ng_pass_max_completed_air_distance` | Extreme-value metric, high variance, low predictive value |
| `ng_pass_max_air_distance` | Same issue |

**Keep — tracking quality metrics (unique to NextGen):**
- Passing: `ng_pass_avg_time_to_throw`, `ng_pass_avg_completed_air_yards`, `ng_pass_avg_intended_air_yards`, `ng_pass_avg_air_yards_differential`, `ng_pass_aggressiveness`, `ng_pass_avg_air_yards_to_sticks`, `ng_pass_completion_percentage`, `ng_pass_expected_completion_percentage`, `ng_pass_completion_percentage_above_expectation`, `ng_pass_avg_air_distance`
- Rushing: `ng_rush_efficiency`, `ng_rush_percent_attempts_gte_eight_defenders`, `ng_rush_avg_time_to_los`, `ng_rush_avg_rush_yards`, `ng_rush_expected_rush_yards`, `ng_rush_rush_yards_over_expected`, `ng_rush_rush_yards_over_expected_per_att`, `ng_rush_pct_over_expected`
- Receiving: `ng_rec_avg_cushion`, `ng_rec_avg_separation`, `ng_rec_avg_intended_air_yards`, `ng_rec_percent_share_of_intended_air_yards`, `ng_rec_catch_percentage`, `ng_rec_avg_yac`, `ng_rec_avg_expected_yac`, `ng_rec_avg_yac_above_expectation`

**Action:** Remove the 14 duplicate/low-value columns listed above from `loaders/nextgen.py`.

## 7. `pfr_adv_stats`
DECISION: keep all columns for this table - we will analyze and model based on positional groups so the sparse wide dataset is not an issue here

### ✂️ Suggestions — `pfr_adv_stats`

This table is **very wide and sparse** due to 4 role-specific stat groups merged into one row. Most cells are NULL for any given player.

**Drop — `pfr_def_*` group (all 20 columns):**  
Prop betting targets offensive players. Defensive stats create wide null-heavy rows with no value for our use case.

| Drop Group | Count | Reason |
|---|---|---|
| `pfr_def_*` columns | ~20 | Defensive players only; >90% null for offensive player rows |
| `pfr_game_id` | 1 | Redundant with `game_id` |
| `pfr_rec_ints` | 1 | QB metric stored in receiver rows — high null rate, wrong grain |
| `pfr_rec_passer_rating` | 1 | Same issue as above |

**Keep — offensive-focused columns:**
- Identity: `game_id`, `season`, `week`, `game_type`, `team`, `opponent`, `pfr_player_id`, `pfr_player_name`
- Passing: `pfr_pass_drops`, `pfr_pass_drop_pct`, `pfr_pass_bad_throws`, `pfr_pass_bad_throw_pct`, `pfr_pass_times_sacked`, `pfr_pass_times_blitzed`, `pfr_pass_times_hurried`, `pfr_pass_times_hit`, `pfr_pass_times_pressured`, `pfr_pass_times_pressured_pct`
- Rushing: `pfr_rush_carries`, `pfr_rush_ybc`, `pfr_rush_ybc_avg`, `pfr_rush_yac`, `pfr_rush_yac_avg`, `pfr_rush_broken_tackles`
- Receiving: `pfr_rec_broken_tackles`, `pfr_rec_drops`, `pfr_rec_drop_pct`

**Action:** Remove `pfr_def_*`, `pfr_game_id`, `pfr_rec_ints`, `pfr_rec_passer_rating` from `loaders/pfr_adv_stats.py`.

## 8. `player_info`
DECISION: remove the suggested columns and keep table

### ✂️ Suggestions — `player_info`

| Column | Keep? | Reason |
|---|---|---|
| `gsis_id` | ✅ | Primary key |
| `pi_display_name` | ✅ | Human-readable name |
| `pi_position_group` | ✅ | Authoritative position grouping |
| `ngs_position` | ✅ | Fine-grained position (e.g., SLOT_WR) — distinct from group |
| `ngs_position_group` | ❌ | Redundant with `pi_position_group`; keep one |
| `height` | ✅ | Physical attribute |
| `weight` | ✅ | Physical attribute |
| `rookie_season` | ✅ | Seniority/experience context |
| `draft_round` | ✅ | Talent signal |
| `draft_pick` | ✅ | Talent signal |
| `years_of_experience` | ✅ | Computed from rookie_season; check if it adds info |
| `pff_position` | ⚠️ | Check null rate — drop if heavily null or identical to `ngs_position` |
| `college_name` | ❌ | Not useful for in-season game predictions |
| `college_conference` | ❌ | Not useful for in-season game predictions |
| `pfr_id` | ❌ | Already in `fantasy_ids` — pure duplication |

**Action:** Remove `ngs_position_group`, `college_name`, `college_conference`, `pfr_id` from `loaders/player_info.py`.  
Evaluate `pff_position` and `years_of_experience` based on null rates above.

## 9. `play_by_play`
DECISION: drop the suggested columns to drop and keep the advanced columns mentioned

### ✂️ Suggestions — `play_by_play`

The unique PBP value is **EPA (expected points added)** and **situational/red-zone breakdowns**. Basic counting stats duplicate `weekly_player_stats`.

**Drop — duplicate counting stats:**

| Column | Drop Reason |
|---|---|
| `pbp_pass_attempts` | = `weekly_player_stats.passing_attempts` |
| `pbp_completions` | = `weekly_player_stats.completions` |
| `pbp_incompletions` | Derivable from attempts - completions |
| `pbp_pass_yards` | = `weekly_player_stats.passing_yards` |
| `pbp_pass_tds` | = `weekly_player_stats.passing_tds` |
| `pbp_ints` | = `weekly_player_stats.interceptions` |
| `pbp_carries` | = `weekly_player_stats.rushing_attempts` |
| `pbp_rush_yards` | = `weekly_player_stats.rushing_yards` |
| `pbp_rush_tds` | = `weekly_player_stats.rushing_tds` |
| `pbp_targets` | = `weekly_player_stats.targets` |
| `pbp_receptions` | = `weekly_player_stats.receptions` |
| `pbp_rec_yards` | = `weekly_player_stats.receiving_yards` |
| `pbp_rec_tds` | = `weekly_player_stats.receiving_tds` |

**Keep — EPA and advanced metrics (unique to PBP):**
- Pass EPA: `pbp_pass_epa_total`, `pbp_qb_epa_total`, `pbp_air_epa`, `pbp_yac_epa`, `pbp_cpoe_mean`, `pbp_cp_mean`, `pbp_xyac_epa`, `pbp_xpass_mean`, `pbp_pass_oe_mean`
- Rush EPA: `pbp_rush_epa_total`, `pbp_xyac_rush_epa`
- Receiving EPA: `pbp_rec_epa_total`, `pbp_rec_air_epa`, `pbp_rec_yac_epa`, `pbp_xyac_rec_mean`
- Situational: `pbp_rz_dropbacks`, `pbp_rz_carries`, `pbp_rz_targets`, `pbp_deep_targets`, `pbp_goal_line_carries`
- Air yards: `pbp_air_yards_thrown`, `pbp_rec_air_yards`, `pbp_rec_yac`
- Context: `pbp_sacks`, `pbp_scrambles`, `pbp_nohuddle_snaps`, `pbp_shotgun_snaps`, `pbp_first_down_pass`, `pbp_rush_first_downs`, `pbp_rec_first_downs`, `pbp_tfl`

**Action:** Remove the 13 duplicate counting stat columns listed above from `loaders/play_by_play.py`.

## 10. `rosters`
DECISION: keep suggested columns

### ✂️ Suggestions — `rosters`

| Column | Keep? | Reason |
|---|---|---|
| `season` | ✅ | Join key |
| `week` | ✅ | Join key |
| `game_type` | ✅ | REG/POST filter |
| `gsis_id` | ✅ | Join key |
| `team` | ✅ | Current team (weekly) |
| `ros_position` | ✅ | Roster position (may differ from depth_chart) |
| `ros_status` | ✅ | Active/IR/PUP — availability signal |
| `ros_status_desc` | ✅ | Detailed status description |
| `ros_depth_chart_pos` | ❌ | Covered at finer grain by `depth_chart` loader |
| `ros_height` | ❌ | Already in `player_info.height` — pure duplication |
| `ros_weight` | ❌ | Already in `player_info.weight` — pure duplication |

**Action:** Remove `ros_depth_chart_pos`, `ros_height`, `ros_weight` from `loaders/rosters.py`.

## 11. `schedule`
DECISION: keep suggested columns

### ✂️ Suggestions — `schedule`

| Column | Keep? | Reason |
|---|---|---|
| `game_id` | ✅ | Primary join key |
| `season` | ✅ | Filter |
| `week` | ✅ | Filter |
| `game_type` | ✅ | REG/POST |
| `gameday` | ✅ | Date context |
| `gametime` | ✅ | Time of day context (early/late/night) |
| `location` | ✅ | Home/Away/Neutral |
| `result` | ✅ | Target for game-level modeling |
| `total` | ✅ | Actual total points |
| `spread_line` | ✅ | Market spread |
| `total_line` | ✅ | Market O/U |
| `away_moneyline` | ✅ | Raw moneyline (keep both for context) |
| `home_moneyline` | ✅ | Raw moneyline |
| `team_moneyline` | ✅ | Per-team moneyline (reshaped) |
| `spread_from_team` | ✅ | Per-team spread (reshaped) |
| `div_game` | ✅ | Division rivalry signal |
| `roof` | ✅ | Weather context |
| `surface` | ✅ | Field surface |
| `temp` | ✅ | Temperature |
| `wind` | ✅ | Wind speed |
| `stadium` | ✅ | Human-readable venue |
| `team` | ✅ | Per-team row key |
| `opponent_team` | ✅ | Opponent |
| `team_score` | ✅ | Actual points scored |
| `opp_score` | ✅ | Opponent points |
| `rest_days` | ✅ | Key prep-time signal |
| `team_coach` | ✅ | Coach effect signal |
| `is_home` | ✅ | Home/away binary |
| `pfr` | ❌ | Alternative game ID — not used in joins |
| `espn` | ❌ | Alternative game ID — not used in joins |
| `old_game_id` | ❌ | Legacy ID — not used |
| `stadium_id` | ❌ | Redundant with `stadium` name |
| `referee` | ❌ | Weak signal; high cardinality |

**Action:** Remove `pfr`, `espn`, `old_game_id`, `stadium_id`, `referee` from `loaders/schedule.py`.

## 12. `snap_counts`
DECISION: keep suggested columns

### ✂️ Suggestions — `snap_counts`

| Column | Keep? | Reason |
|---|---|---|
| `game_id` | ✅ | Join key |
| `season` | ✅ | Filter |
| `week` | ✅ | Filter |
| `game_type` | ✅ | REG/POST |
| `pfr_player_id` | ✅ | Primary key for this table |
| `team` | ✅ | Context |
| `offense_snaps` | ✅ | Raw snap count |
| `offense_pct` | ✅ | Snap share — key feature |
| `defense_snaps` | ✅ | For defensive players |
| `defense_pct` | ✅ | Defensive snap share |
| `st_snaps` | ✅ | Special teams count |
| `st_pct` | ✅ | Special teams share |
| `snap_player_name` | ❌ | Redundant with `player_info.pi_display_name` |
| `snap_position` | ❌ | Redundant with `player_info`/`rosters` position columns |

**Action:** Remove `snap_player_name`, `snap_position` from `loaders/snap_counts.py`.

## 13. `weekly_player_stats`
DECISION: keep suggested columns

### ✂️ Suggestions — `weekly_player_stats`

This loader already does substantial column pruning. Remaining suggestions:

| Column | Keep? | Reason |
|---|---|---|
| `def_tackles_combined` | ✅ | = solo + assist; the composite metric |
| `def_tackles_solo` | ❌ | Component of `def_tackles_combined` — drop to reduce width |
| `def_tackles_assist` | ❌ | Component of `def_tackles_combined` — drop to reduce width |
| `def_tackles` | ⚠️ | Verify: is this solo-only or combined? If same as `def_tackles_combined`, drop |
| `receiving_yards_after_catch` | ❌ | Already in `play_by_play` as `pbp_rec_yac` at higher fidelity |
| All other columns | ✅ | Already pruned — no further redundancy |

**Action:** Remove `def_tackles_solo`, `def_tackles_assist`, `receiving_yards_after_catch` from `loaders/weekly_player_stats.py`.  
Verify `def_tackles` vs `def_tackles_combined` and drop whichever is redundant.

## 14. `weekly_team_stats`
DECISION: keep suggested columns

### ✂️ Suggestions — `weekly_team_stats`

**General principle:** This table provides **team-level efficiency context** that can't be aggregated from player stats (e.g., `yards_per_play`, `time_of_possession`, `first_downs`). Pure sum-level duplicates of player stats can be dropped.

Based on the column overlap printed above:
- **Drop** any column that is just a sum of `weekly_player_stats` columns (e.g., `passing_yards`, `rushing_tds`) — these are redundant when you can aggregate player-level data.
- **Keep** efficiency/rate columns unique to team-level data: `yards_per_play`, `time_of_possession`, `third_down_pct`, `red_zone_attempts`, `first_downs`, etc.
- **Keep** defensive team columns (no player-level equivalent).
- **Keep** turnover/penalty aggregates that aren't in player stats.

**Action:** Review the overlap list above and remove duplicate sum-level columns from `loaders/weekly_team_stats.py`.

---
## Summary of Recommended Changes

| Loader | Columns to Drop | Rationale |
|---|---|---|
| `fantasy_ids` | `sportradar_id`, `espn_id`, `sleeper_id`, `yahoo_id`, `mfl_id` | Platform IDs unused downstream |
| `fantasy_rankings` | `rank_player_owned_espn`, `rank_player_owned_yahoo`, `rank_bye` | Redundant ownership + draft-only field |
| `nextgen` | 14 counting stats (see section 6) | Duplicate `weekly_player_stats`; keep tracking metrics |
| `pfr_adv_stats` | All `pfr_def_*`, `pfr_game_id`, `pfr_rec_ints`, `pfr_rec_passer_rating` | Defensive data + duplicates |
| `player_info` | `ngs_position_group`, `college_name`, `college_conference`, `pfr_id` | Redundant or non-predictive |
| `play_by_play` | 13 counting stats (see section 9) | Duplicate `weekly_player_stats`; keep EPA metrics |
| `rosters` | `ros_depth_chart_pos`, `ros_height`, `ros_weight` | Covered elsewhere |
| `schedule` | `pfr`, `espn`, `old_game_id`, `stadium_id`, `referee` | Unused IDs + weak signal |
| `snap_counts` | `snap_player_name`, `snap_position` | Covered by `player_info` |
| `weekly_player_stats` | `def_tackles_solo`, `def_tackles_assist`, `receiving_yards_after_catch` | Components of combined columns + PBP duplicate |
| `weekly_team_stats` | Overlap columns with `weekly_player_stats` | See section 14 output |
| `depth_chart` | None | Already minimal |
| `fantasy_opportunities` | Actual counting stats (see section 3) | Duplicate `weekly_player_stats` |
| `formations` | None | Already minimal |